# DuckDB + SLayer, from the command line

This shows how to use SLayer with DuckDB entirely from the command line, from scratch. 

We install the DuckDB CLI, expose a **48 KB CSV on a CDN** as a DuckDB view, let SLayer **auto-ingest** its schema into a dataset, and query it with `slayer query`. Nothing is copied locally — the view points at the URL and every query reaches back over the wire.

Each `slayer query` runs in a shell cell, outputting results as json. For readability, we use pandas to display it.

**Prerequisites:** `pip install motley-slayer`.

## 1. Set it up — all CLI

Three steps, no Python: install the DuckDB CLI, create a **view** over the remote CSV inside a DuckDB file, then point a SLayer datasource at that file and let `--ingest` **auto-build the dataset** by introspecting the view. Column names and types come straight from the data — nothing hand-written.

In [ ]:
%%bash
set -euo pipefail
export SLAYER_STORAGE=.cache/cli/store
rm -rf .cache/cli && mkdir -p .cache/cli

# Install the DuckDB CLI if it isn't already on PATH (idempotent). The installer
# uses bash syntax, so pipe it to bash — /bin/sh is dash on many CI runners.
command -v duckdb >/dev/null 2>&1 || curl -fsSL https://install.duckdb.org | bash >/dev/null
export PATH="$HOME/.duckdb/cli/latest:$PATH"

# Expose the remote CSV as a DuckDB view: the CSV stays on the CDN, the view
# just points at it over httpfs — nothing is copied locally.
duckdb .cache/cli/weather.duckdb -c \
  "CREATE VIEW weather AS SELECT * FROM read_csv_auto('https://cdn.jsdelivr.net/npm/vega-datasets@2/data/seattle-weather.csv')"

# Register the datasource and auto-ingest the view into a dataset —
# column names and types come from introspecting the view, nothing hand-written.
PYTHONWARNINGS=ignore slayer datasources create "duckdb:///$PWD/.cache/cli/weather.duckdb" \
  --name weather_db --ingest -y
slayer datasets list

## 2. A warm-up query

A plain grouped count — how many days of each weather type. The shell cell runs `slayer query ... --format json` and captures its output; the next line renders it with pandas.

In [ ]:
%%bash --out warmup
export SLAYER_STORAGE=.cache/cli/store
slayer query '{"population": "weather", "dimensions": ["weather.weather"], "metrics": [{"metric": "count(weather.*)", "alias": "days"}]}' --format json

In [ ]:
import json

import pandas as pd
from IPython.display import Markdown

pd.DataFrame(json.loads(warmup))

## 3. The hero query: an aggregate as a dimension, plus a ranking transform

One query, single stage, two query-time features at the month grain:

- **A dimension computed from an aggregate.** `season` is `CASE WHEN avg(weather.temp_max, partition_by=weather.date) >= 18 THEN 'warm' ELSE 'cool' END` — each month is grouped *warm* or *cool* by its own average high, a value that only exists after aggregating. SLayer computes it in a synthesized stage and regroups on it.
- **A ranking transform in a metric.** `rank(sum(weather.precipitation), direction='desc')` orders the months by rainfall, 1 = wettest.

Ordered wettest-first, the answer tells a story: Seattle's rainiest months are all *cool*-season.

We write the query to a file and run it with `slayer query @file` — the same JSON the Python notebook hands to the client.

In [ ]:
%%bash --out hero_rows
set -e
export SLAYER_STORAGE=.cache/cli/store
cat > .cache/cli/hero.json <<'JSON'
{
  "population": "weather",
  "time_dimensions": [{"field": "weather.date", "granularity": "month"}],
  "dimensions": [
    {"field": "CASE WHEN avg(weather.temp_max, partition_by=weather.date) >= 18 THEN 'warm' ELSE 'cool' END", "alias": "season"}
  ],
  "metrics": [
    {"metric": "sum(weather.precipitation)", "alias": "total_rain"},
    {"metric": "rank(sum(weather.precipitation), direction='desc')", "alias": "rain_rank"}
  ],
  "order": [{"field": "rain_rank", "direction": "asc"}]
}
JSON
slayer query @.cache/cli/hero.json --format json

In [ ]:
pd.DataFrame(json.loads(hero_rows))

## 4. The SQL SLayer generated

But why would you want to use SLayer at all, rather than call DuckDB directly using SQL? 

This cell shows why. Compare the simple, natural json query above and the SQL below - which of them is an agent more likely to write correctly, every time?

In [ ]:
%%bash --out hero_sql
export SLAYER_STORAGE=.cache/cli/store
slayer query @.cache/cli/hero.json --dry-run

In [ ]:
Markdown("```sql\n" + hero_sql + "\n```")

---

The whole semantic layer over a file on the internet — a DuckDB view, auto-ingestion, and JSON queries, all from the shell. See the [Python notebook](duckdb_python_nb.ipynb) for the in-process library version, [aggregations](../07_aggregations/aggregations.md) for more information about SLayer query syntax, and [expressions](../../concepts/expressions.md) for the full transform vocabulary.